# 01 - Pull the public data (EVERYONE, in parallel)

**What this does:** streams a *small* number of real and AI-generated images from public datasets on Hugging Face
and saves them to the shared Drive folder, plus a small CSV describing each image.

**Why it is small now:** the old version asked for thousands of images and kept reading the stream to find rare
generators. That is what crashed the laptops. This version has hard limits (images, minutes, storage) and stops by itself.

| Member | Pulls (images per class, so 100 means 100 real + 100 AI) |
|---|---|
| GW | OpenFake 150 |
| AW | OpenFake 100 |
| NG | OpenFake 150 + Reddit (in-the-wild test set) 150 |
| NC | OpenFake 100 + Community Forensics 100 |
| SM | Community Forensics 100 |
| PN | OpenFake 100 + Community Forensics 100 |

**Rules:** set `MEMBER` to your own code. Run cells one at a time (never "Run all"). Do the PILOT first (it takes about 3 minutes),
then set `PILOT = False`. If Colab disconnects, just run the cells again: it resumes.

In [1]:
# CELL 1 - run this first. It downloads the project code into this Colab session.
import os, subprocess, sys
REPO_URL = "https://github.com/andrinawatsemba/DEEPFAKE_CV_INVESTIGATION.git"
CODE = "/content/repo"
if not os.path.isdir(CODE + "/src"):
    for branch in ("main", "gw/lean-pipeline"):          # main first; the branch is only a fallback
        subprocess.run(["rm", "-rf", CODE])
        r = subprocess.run(["git", "clone", "--depth", "1", "-b", branch, REPO_URL, CODE])
        if r.returncode == 0 and os.path.exists(CODE + "/src/colab_setup.py"):
            print("Got the code from branch:", branch)
            break
else:
    subprocess.run(["git", "-C", CODE, "pull", "--ff-only"])
sys.path.insert(0, CODE + "/src")
print("Code ready.")


Got the code from branch: main
Code ready.


In [2]:
# CELL 2 - install (about 1 minute)
!pip -q install -U datasets imagehash psutil

In [8]:
# CELL 3 - EDIT THESE TWO LINES ONLY
MEMBER = "AW"      # your code: GW, AW, NG, NC, SM or PN
PILOT = False      # True = tiny test. After it works, change to False and run again.

In [9]:
# CELL 4 - connect Drive and check memory
from colab_setup import setup
ROOT, REPO = setup()
try:
    import psutil
    m = psutil.virtual_memory()
    print(f"RAM free: {m.available/1e9:.1f} GB of {m.total/1e9:.1f} GB")
except Exception:
    pass
try:                                                    # optional: faster downloads if you saved HF_TOKEN in Colab Secrets
    from google.colab import userdata
    import os
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("Using your Hugging Face token.")
except Exception:
    print("No HF_TOKEN secret found (that is fine for public data).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project data folder: /content/drive/MyDrive/Deepfake Detection Project/datasets
Code folder:         /content/repo
RAM free: 11.6 GB of 13.6 GB
Using your Hugging Face token.


In [10]:
# CELL 5 - pull (this is the main step)
import pull_data as P
results = P.run_member(MEMBER, ROOT, pilot=PILOT)


=== openfake: 100 real + 100 AI for AW (FULL) ===


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Resolving data files:   0%|          | 0/608 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/608 [00:00<?, ?it/s]

[openfake] resuming: 16 images already saved by AW ({'real': 8, 'ai': 8})
[openfake] columns: ['image', 'image_sha256', 'label', 'model', 'prompt', 'prompt_kind', 'prompt_model', 'prompt_revision', 'prompt_version', 'release_date', 'row_id', 'type']
[openfake] saved {'real': 27, 'ai': 23} | looked at 50 | 66 MB | 17 s
[openfake] saved {'real': 61, 'ai': 39} | looked at 100 | 135 MB | 20 s
[openfake] saved {'real': 72, 'ai': 78} | looked at 172 | 177 MB | 23 s
[openfake] saved {'real': 100, 'ai': 100} | looked at 596 | 203 MB | 32 s
{'source': 'openfake', 'member': 'AW', 'counts': {'real': 100, 'ai': 100}, 'looked_at': 597, 'megabytes': 203, 'seconds': 37, 'ai_groups_seen': 36, 'stopped_because': 'target reached', 'skipped': {'duplicate': 16, 'group_cap': 189, 'class_full': 207}}


In [11]:
# CELL 6 - what did I get?  Check: both classes present, several generators, a mix of formats.
P.summary(ROOT, MEMBER)

public_openfake_AW.csv           rows=  200  {'real': np.int64(100), 'ai': np.int64(100)}  generators=36  formats={'JPEG': np.int64(128), 'PNG': np.int64(67), 'WEBP': np.int64(5)}


## Share your result with the group (GitHub, no images)

Images stay on Drive (they are large and licence-restricted). What goes on GitHub is the small CSV that describes them.
Run the next cell, then upload the downloaded file(s):
GitHub repo > `metadata/` folder > **Add file > Upload files**.
**Only do this after the FULL pull (`PILOT = False`).**

In [12]:
# CELL 7 - download your CSV file(s) to upload to GitHub
import glob, os, shutil
from google.colab import files
mine = sorted(glob.glob(os.path.join(ROOT, "metadata", f"public_*_{MEMBER}.csv")))
print("Files:", [os.path.basename(f) for f in mine])
for f in mine:
    files.download(f)

Files: ['public_openfake_AW.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>